# 🤟 Sign Language Detection — Training & Analysis

**Internship Task Extension:** Sign Language Detection with ASL alphabet + 10 custom words

## System
- **Hand detection**: MediaPipe (21 landmarks)
- **Classification**: Dense neural network on 63 landmark features
- **Labels**: A-Z (26) + Hello, ThankYou, Yes, No, Please, Sorry, Help, Eat, Water, More (10)
- **Time window**: 6:00 PM – 10:00 PM only


In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
!pip install mediapipe tensorflow opencv-python scikit-learn \
             matplotlib seaborn numpy pillow -q
print('✓ Dependencies installed')

In [ ]:
# ── CELL 2: Setup ─────────────────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
import mediapipe as mp
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('samples', exist_ok=True)
print('✓ Setup done')

In [ ]:
# ── CELL 3: MediaPipe landmark extraction demo ────────────────────────────────
sys.path.insert(0, '.')
from config import LANDMARK_FEATURES, ALL_LABELS, KNOWN_WORDS, ALPHABET

print(f'Total labels: {len(ALL_LABELS)}')
print(f'  Words ({len(KNOWN_WORDS)}): {KNOWN_WORDS}')
print(f'  Alphabet ({len(ALPHABET)}): {" ".join(ALPHABET)}')
print(f'\nFeature vector size: {LANDMARK_FEATURES} (21 landmarks × 3 coords)')

# Visualise the 21 landmark positions
fig, ax = plt.subplots(figsize=(6, 8))

# MediaPipe hand landmark skeleton
connections = [
    (0,1),(1,2),(2,3),(3,4),        # thumb
    (0,5),(5,6),(6,7),(7,8),        # index
    (0,9),(9,10),(10,11),(11,12),   # middle
    (0,13),(13,14),(14,15),(15,16), # ring
    (0,17),(17,18),(18,19),(19,20), # pinky
    (5,9),(9,13),(13,17),           # palm
]

# Approximate landmark positions for visualization
lm_positions = [
    (0.5, 0.0),    # 0 wrist
    (0.22, 0.22),  # 1 thumb CMC
    (0.1, 0.38),   # 2 thumb MCP
    (0.03, 0.52),  # 3 thumb IP
    (-0.03, 0.65), # 4 thumb tip
    (0.35, 0.45),  # 5 index MCP
    (0.33, 0.62),  # 6 index PIP
    (0.32, 0.76),  # 7 index DIP
    (0.31, 0.88),  # 8 index tip
    (0.5, 0.47),   # 9 middle MCP
    (0.5, 0.65),   # 10 middle PIP
    (0.5, 0.79),   # 11 middle DIP
    (0.5, 0.91),   # 12 middle tip
    (0.65, 0.46),  # 13 ring MCP
    (0.67, 0.63),  # 14 ring PIP
    (0.68, 0.76),  # 15 ring DIP
    (0.68, 0.87),  # 16 ring tip
    (0.78, 0.42),  # 17 pinky MCP
    (0.82, 0.57),  # 18 pinky PIP
    (0.84, 0.69),  # 19 pinky DIP
    (0.85, 0.80),  # 20 pinky tip
]

for i, j in connections:
    x0, y0 = lm_positions[i]
    x1, y1 = lm_positions[j]
    ax.plot([x0, x1], [y0, y1], '-', color='#00e5ff', alpha=0.6, lw=2)

for idx, (x, y) in enumerate(lm_positions):
    col = '#ff9800' if idx == 0 else '#76ff03'
    ax.scatter(x, y, c=col, s=80, zorder=5)
    ax.annotate(str(idx), (x, y), fontsize=7, ha='center', va='bottom',
                color='white', xytext=(0, 5), textcoords='offset points')

ax.set_xlim(-0.2, 1.1)
ax.set_ylim(-0.1, 1.1)
ax.set_aspect('equal')
ax.set_title('MediaPipe Hand — 21 Landmarks', fontsize=12)
ax.invert_yaxis()
ax.axis('off')
plt.tight_layout()
plt.savefig('outputs/hand_landmarks.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/hand_landmarks.png')

In [ ]:
# ── CELL 4: Download ASL dataset (Kaggle) ────────────────────────────────────
# Option A: Use kaggle API
# !pip install kaggle -q
# !kaggle datasets download -d grassknoted/asl-alphabet
# !unzip asl-alphabet.zip -d dataset/

# Option B: Use a small demo subset from GitHub
print('Dataset setup options:')
print('  A) Kaggle: kaggle datasets download grassknoted/asl-alphabet')
print('  B) Manual: Upload your own hand images to dataset/train/<label>/')
print('  C) Capture: python train.py --capture')
print()
print('Expected structure:')
print('  dataset/train/')
print('      A/ *.jpg')
print('      B/ *.jpg')
print('      Hello/ *.jpg')
print('      ...')

In [ ]:
# ── CELL 5: Train the model ───────────────────────────────────────────────────
# Run after setting up dataset in Cell 4
!python train.py --epochs 50 --batch 32

In [ ]:
# ── CELL 6: Test on uploaded image ───────────────────────────────────────────
from google.colab import files
from utils.detector import SignLanguageDetector
import matplotlib.pyplot as plt
import cv2

print('Upload a hand sign image...')
uploaded = files.upload()

detector = SignLanguageDetector()

for fname, data in uploaded.items():
    path = f'samples/{fname}'
    with open(path, 'wb') as f:
        f.write(data)

    # Use --demo flag equivalent: bypass time window
    result = detector.predict_image(path, min_confidence=0.5)

    print(f'\n{"═"*40}')
    print(f'  Sign       : {result.get("prediction", "—")}')
    print(f'  Confidence : {result.get("confidence", 0):.1%}')
    print(f'  Hands      : {result.get("hands_detected", 0)}')
    print(f'{"═"*40}')

    if result.get('annotated') is not None:
        rgb = cv2.cvtColor(result['annotated'], cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 6))
        plt.imshow(rgb)
        plt.axis('off')
        plt.title(f"Prediction: {result['prediction']}  ({result['confidence']:.1%})")
        plt.savefig(f'outputs/result_{fname}', dpi=150)
        plt.show()

In [ ]:
# ── CELL 7: Show training plots ───────────────────────────────────────────────
import os
from IPython.display import Image as IPyImage, display

for plot_name in ['training_history.png', 'confusion_matrix.png']:
    path = f'models/{plot_name}'
    if os.path.exists(path):
        print(f'--- {plot_name} ---')
        display(IPyImage(path))
    else:
        print(f'  {plot_name} not found — train the model first (Cell 5)')

In [ ]:
# ── CELL 8: Model architecture summary ───────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

n_classes = 36  # 26 letters + 10 words

model = keras.Sequential([
    layers.Input(shape=(63,)),
    layers.Dense(256), layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.4),
    layers.Dense(128), layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.3),
    layers.Dense(64, activation='relu'), layers.Dropout(0.2),
    layers.Dense(n_classes, activation='softmax'),
], name='sign_language_nn')

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

total_params = model.count_params()
print(f'\nTotal parameters: {total_params:,}')
print(f'Input features:   63 (21 landmarks × 3 coords)')
print(f'Output classes:   {n_classes}')

In [ ]:
# ── CELL 9: Time window verification ─────────────────────────────────────────
from datetime import datetime, time as dtime
from config import ACTIVE_START, ACTIVE_END

now = datetime.now()
is_active = ACTIVE_START <= now.time() <= ACTIVE_END

print(f'Operating window : {ACTIVE_START.strftime("%I:%M %p")} – {ACTIVE_END.strftime("%I:%M %p")}')
print(f'Current time     : {now.strftime("%I:%M %p")}')
print(f'System active    : {"✓ YES" if is_active else "✗ NO (use --demo to bypass)"}')

# Visualise time window
fig, ax = plt.subplots(figsize=(12, 2))
ax.barh(0, 24, color='#1a1a2e', height=0.6)
ax.barh(0, ACTIVE_END.hour - ACTIVE_START.hour,
        left=ACTIVE_START.hour, color='#00e5ff', height=0.6, alpha=0.8)
ax.axvline(x=now.hour + now.minute/60, color='#ff3d00', lw=2, label='Now')
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 2))
ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 25, 2)], rotation=30)
ax.set_yticks([])
ax.set_title(f'Operating Window: {ACTIVE_START.strftime("%I:%M %p")} – {ACTIVE_END.strftime("%I:%M %p")} (cyan = active)', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('outputs/time_window.png', dpi=150)
plt.show()

In [ ]:
# ── CELL 10: Download outputs ─────────────────────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('sign_language_outputs', 'zip', 'outputs')
files.download('sign_language_outputs.zip')
print('Downloaded outputs.zip')